# 改变 awkward 数组的形状 ― flatten 与 unflatten

## 本章要学什么

awkward 的数组可以随意改变层次的数量以及各轴的长度，这一章用实际的数组确认把层次压平的用法和增加层次的办法。

* `flatten()` 与 `ravel()` 的区别
* 固定长的轴(regular)与长度会变的轴(var)之间的转换
* 先把长度对齐再变成固定长的方法
* 用 `unflatten()` 增加层次的方法
* 数组的连接，以及记录(结构体)的操作

In [1]:
import numpy as np
import awkward as ak

## flatten 与 ravel

先用多层嵌套的 Awkward Array，确认 `flatten()` 会把哪个轴、以什么方式压平。

In [2]:
a = ak.Array([[[1, 2], None, [3]], [[None, 4], [5]], None, [[6, None]]])
a

<Array [[[1, 2], None, [3]], ..., [[6, ...]]] type='4 * option[var * option...'>

Awkward Array 在 `axis=0` 上做 flatten 时，只是把最上层的 None 去掉。

In [3]:
ak.flatten(a, axis=0)    # 只去掉最上层的 None

<Array [[[1, 2], None, [3]], ..., [[6, ...]]] type='3 * var * option[var * ...'>

接着指定 `axis=1` 或 `axis=2`，对应的那一层会被删掉，元素则移动到浅一层的层级里。

In [4]:
ak.flatten(a, axis=1)    # 压平下一层的层次

<Array [[1, 2], None, [3], ..., [5], [6, None]] type='6 * option[var * ?int64]'>

In [5]:
ak.flatten(a, axis=2)    # 压平最内层的层次

<Array [[1, 2, 3], [None, ...], ..., [6, None]] type='4 * option[var * ?int64]'>

想删掉多个层次时，就重复使用 `flatten`。

In [6]:
ak.flatten(ak.flatten(a, axis=2), axis=1)   # 一次压平两层

<Array [1, 2, 3, None, 4, 5, 6, None] type='8 * ?int64'>

指定 `axis=None` 时，最后一轴的所有值都被取出来排成一维，而 `None` 会被去掉。

In [7]:
ak.flatten(a, axis=None)   # 变成一维，丢掉 None

<Array [1, 2, 3, 4, 5, 6] type='6 * int64'>

另一方面，`ravel()` 与 `flatten(None)` 相似，但它会保留 `None`。

In [8]:
ak.ravel(a)              # 变成一维，但保留 None

<Array [1, 2, 3, None, 4, 5, 6, None] type='8 * ?int64'>

## 规则轴与非规则轴

接下来确认由 NumPy 数组生成的 Awkward Array 的规则轴(Regular)，与由列表生成的非规则轴(Irregular)之间的差别。

In [9]:
np.random.seed(42)
arr = np.random.randint(0, 10, (4, 6))
a = ak.Array(arr)
a

<Array [[6, 3, 7, 4, 6, 9], ..., [5, 1, 4, 0, 9, 5]] type='4 * 6 * int32'>

In [10]:
b = ak.Array(arr.tolist())   # 来自列表，所以轴是可变长(var)
b

<Array [[6, 3, 7, 4, 6, 9], ..., [5, 1, 4, ..., 9, 5]] type='4 * var * int64'>

把固定长的轴变成可变长。

In [11]:
ak.from_regular(a, axis=1)   # 把固定长的轴变成可变长

<Array [[6, 3, 7, 4, 6, 9], ..., [5, 1, 4, ..., 9, 5]] type='4 * var * int32'>

把可变长的轴变成固定长。

In [12]:
ak.to_regular(b, axis=1)     # 把可变长的轴变成固定长

<Array [[6, 3, 7, 4, 6, 9], ..., [5, 1, 4, 0, 9, 5]] type='4 * 6 * int64'>

规则轴上的布尔索引与 NumPy 完全一样。

In [13]:
arr[arr >= 4]

array([6, 7, 4, 6, 9, 6, 7, 4, 7, 7, 5, 4, 7, 5, 4, 9, 5], dtype=int32)

In [14]:
a[a >= 4]

<Array [6, 7, 4, 6, 9, 6, 7, 4, ..., 7, 5, 4, 7, 5, 4, 9, 5] type='17 * int32'>

在非规则轴上做筛选时，嵌套结构会被原样保留。

In [15]:
c = b[b >= 4]
c

<Array [[6, 7, 4, 6, 9], [6, ..., 7], ..., [5, 4, 9, 5]] type='4 * var * int64'>

由于是非规则轴，直接 `to_regular()` 会报错。

In [16]:
try:
    ak.to_regular(c)
except ValueError as ex:
    print(ex)

cannot convert to RegularArray because subarray lengths are not regular (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-57/awkward-cpp/src/cpu-kernels/awkward_ListOffsetArray_toRegularArray.cpp#L22)


### 先对齐长度再规则化

下面的代码把长度不一的部分子数组补齐到最长的那个：先用 `ak.num(c)` 求出各子数组的长度，再用 `ak.max()` 取最大值，然后用这个长度调用 `ak.pad_none(c, target_length)`，缺少的部分会在末尾补上 `None`，于是所有子数组长度一致；对齐之后再调用 `ak.to_regular()`，就可以转换成固定长的轴。

In [17]:
d = ak.pad_none(c, ak.max(ak.num(c)))
d

<Array [[6, 7, 4, 6, 9], ..., [5, 4, 9, 5, None]] type='4 * var * ?int64'>

In [18]:
ak.to_regular(d)

<Array [[6, 7, 4, 6, 9], [...], ..., [5, 4, 9, 5, None]] type='4 * 5 * ?int64'>

想在左边补 None 时：

In [19]:
ak.pad_none(c[:, ::-1], ak.max(ak.num(c)))[:, ::-1]   # 在左边补 None

<Array [[6, 7, 4, 6, 9], ..., [None, 5, 4, 9, 5]] type='4 * var * ?int64'>

把 `clip=True` 设上之后，超过指定长度的部分就会被截掉：下面的代码按 `c` 的最小长度来对齐，因此过长的子数组末尾会被丢弃，而截断之后的轴就变成固定长的轴。

In [20]:
ak.pad_none(c, int(ak.min(ak.num(c))), clip=True)   # 过长的部分截掉

<Array [[6, 7, 4, 6], [6, 7, ..., 7], ..., [5, 4, 9, 5]] type='4 * 4 * ?int64'>

### 增加新的规则轴

与 NumPy 一样，可以用 `None` 增加新的轴，这时新增轴的长度为 1，而且是规则轴。

In [21]:
b[..., None]           # 在末尾加上长度为 1 的轴

<Array [[[6], [3], [7], [4], [6], [9]], ..., [...]] type='4 * var * 1 * int64'>

In [22]:
b[:, None, :]          # 在中间加上长度为 1 的轴

<Array [[[6, 3, 7, 4, 6, 9]], ..., [[5, 1, ..., 5]]] type='4 * 1 * var * int64'>

用 `singletons()` 可以在指定轴之后加上一条长度为 1 的非规则轴。

In [23]:
ak.singletons(a, axis=1)   # 把每个元素变成长度为 1 的列表

<Array [[[6], [3], [7], [4], [6], [9]], ..., [...]] type='4 * 6 * var * int32'>

In [24]:
ak.singletons(a, axis=0)   # 把每一行变成长度为 1 的列表

<Array [[[6, 3, 7, 4, 6, 9]], ..., [[5, 1, ..., 5]]] type='4 * var * 6 * int32'>

## 用 unflatten 增加层次

`ak.unflatten()` 把一维数组按指定的长度切分，转换成有层次的列表(部分子数组)。例如下面的代码会把前 3 个元素 `[1, 2, 3]` 作为第一个子数组、接下来的 2 个元素 `[4, 5]` 作为第二个、长度为 0 的空数组作为第三个、最后 3 个元素 `[6, 7, 8]` 作为第四个，分别以嵌套数组的形式返回。这样一来，`unflatten` 就能把一维数组按任意长度分层，做出新的层次结构，后续处理时也能方便地以子数组为单位操作。



In [25]:
a = ak.Array([1, 2, 3, 4, 5, 6, 7, 8])
b = ak.unflatten(a, [3, 2, 0, 3])
b

<Array [[1, 2, 3], [4, 5], [], [6, 7, 8]] type='4 * var * int64'>

已经存在的层次里同样可以指定 `axis` 插入新的层。`ak.unflatten()` 的第 2 个参数是按顺序排列各段长度的二维数组：下面例子里的 `a` 是二维数组(每行是一个子数组)，指定 `axis=1` 就会在每一行内部再切分，按 `[4, 2, 1, 2, 2, 1, 3]` 这个长度序列，第 1 行 `[1,2,3,4,5,6]` 被分成 `[1,2,3,4]` 与 `[5,6]`，第 2 行 `[1,2,3,4,5]` 被分成 `[1]` 与 `[2,3]`，第 3 行 `[1,2,3,4]` 被分成 `[2,1,3]`。

In [26]:
a = ak.Array([[1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5], [1, 2, 3, 4]])
ak.unflatten(a, [4, 2, 1, 2, 2, 1, 3], axis=1)

<Array [[[1, 2, 3, 4], [5, 6]], ..., [[1], ...]] type='3 * var * var * int64'>

在第 0 轴上增加新层次的例子：

In [27]:
ak.unflatten(a, [1, 2], axis=0)   # 在第 0 轴上插入新的层次

<Array [[[1, 2, 3, 4, 5, 6]], [[...], ...]] type='2 * var * var * int64'>

## 数组的连接

用 `concatenate()` 可以沿某个轴连接数组。

默认(axis=0)的连接：

In [28]:
a = ak.Array([1, 2, 3, 4])
b = ak.Array([2, 3, 4, 5])
ak.concatenate([a, b])   # 沿第 0 轴连接

<Array [1, 2, 3, 4, 2, 3, 4, 5] type='8 * int64'>

沿第 1 轴连接：

In [29]:
ak.concatenate([a[:, None], b[:, None]], axis=1)   # 沿第 1 轴连接

<Array [[1, 2], [2, 3], [3, 4], [4, 5]] type='4 * 2 * int64'>

数据类型不同时，会作为 union 型连接起来。

In [30]:
c = ak.Array([[1, 2, 3], [3, 4]])
ak.concatenate([a, c])   # 类型不同，所以成为 union

<Array [1, 2, 3, 4, [1, 2, 3], [3, 4]] type='6 * union[int64, var * int64]'>

把带规则轴与带非规则轴的数组连接起来，结果是非规则轴。

In [31]:
ak.concatenate([a[:, None], c])   # 规则轴与可变长混在一起时变成可变长

<Array [[1], [2], [3], [4], [1, 2, 3], [3, 4]] type='6 * var * int64'>

In [32]:
ak.concatenate([c, a[:2, None]], axis=1)

<Array [[1, 2, 3, 1], [3, 4, 2]] type='2 * var * int64'>

## 结构体数组

字典的列表、列表的字典，两者都会被当作结构体数组处理。

In [33]:
a = ak.Array({"x": [1, 2, 3, 4], "y": [2, 3, 4, 5]})
b = ak.Array(
    [
        {"vx": 10, "vy": 20},
        {"vx": 11, "vy": 21},
        {"vx": 12, "vy": 22},
        {"vx": 13, "vy": 23},
    ]
)
a

<Array [{x: 1, y: 2}, {...}, ..., {x: 4, y: 5}] type='4 * {x: int64, y: int64}'>

In [34]:
b

<Array [{vx: 10, vy: 20}, ..., {vx: 13, ...}] type='4 * {vx: int64, vy: int64}'>

### 增加与删除字段

In [35]:
a['z'] = 0
a

<Array [{x: 1, y: 2, z: 0}, ..., {x: 4, ...}] type='4 * {x: int64, y: int64...'>

In [36]:
a['z'] = [3, 4, 5, 6]
a

<Array [{x: 1, y: 2, z: 3}, ..., {x: 4, ...}] type='4 * {x: int64, y: int64...'>

像字典那样可以增加新字段；删除字段用 `del a['z']`，而要以非破坏性的方式去掉字段时：

In [37]:
ak.without_field(a, 'z')   # 返回去掉 z 的新数组

<Array [{x: 1, y: 2}, {...}, ..., {x: 4, y: 5}] type='4 * {x: int64, y: int64}'>

增加字段的非破坏性写法：

In [38]:
ak.with_field(a, np.sqrt(a.x**2 + a.y**2), 'r')

<Array [{x: 1, y: 2, z: 3, r: 2.24}, ...] type='4 * {x: int64, y: int64, z:...'>

### 用 unzip 展开字段

以元组输出：

In [39]:
ak.unzip(a)             # 用元组取出字段

(<Array [1, 2, 3, 4] type='4 * int64'>,
 <Array [2, 3, 4, 5] type='4 * int64'>,
 <Array [3, 4, 5, 6] type='4 * int64'>)

以字典输出：

In [40]:
ak.unzip(a, how=dict)   # 用字典取出字段

{'x': <Array [1, 2, 3, 4] type='4 * int64'>,
 'y': <Array [2, 3, 4, 5] type='4 * int64'>,
 'z': <Array [3, 4, 5, 6] type='4 * int64'>}

### 用 unzip + zip 合并结构体数组

In [41]:
ak.zip(ak.unzip(a, how=dict) | ak.unzip(b, how=dict))

<Array [{x: 1, y: 2, z: 3, vx: 10, ...}, ...] type='4 * {x: int64, y: int64...'>

### 把结构体数组转换成二维数组

下面说明把结构体数组展开成二维数组的方法。

In [42]:
array = ak.Array(
    [
        {"x": 11, "y": 12, "z": 13},
        {"x": 21, "y": 22, "z": 23},
        {"x": 31, "y": 32, "z": 33},
        {"x": 41, "y": 42, "z": 43},
        {"x": 51, "y": 52, "z": 53},
    ]
)
array

<Array [{x: 11, y: 12, z: 13}, ..., {...}] type='5 * {x: int64, y: int64, z...'>

把一维的记录变成二维：

In [43]:
array[:, None]          # 把每条记录变成长度为 1 的列表

<Array [[{x: 11, y: 12, z: 13}], ..., [{...}]] type='5 * 1 * {x: int64, y: ...'>

把各个字段取出来：

In [44]:
ak.unzip(array[:, None])

(<Array [[11], [21], [31], [41], [51]] type='5 * 1 * int64'>,
 <Array [[12], [22], [32], [42], [52]] type='5 * 1 * int64'>,
 <Array [[13], [23], [33], [43], [53]] type='5 * 1 * int64'>)

把取出的值连起来成为行向量：

In [45]:
ak.concatenate(ak.unzip(array[:, None]), axis=-1)

<Array [[11, 12, 13], [21, ..., 23], ..., [51, 52, 53]] type='5 * 3 * int64'>

用 `singletons()` 也能做同样的处理：

In [46]:
ak.concatenate(ak.unzip(ak.singletons(array)), axis=-1)

<Array [[11, 12, 13], [21, ..., 23], ..., [51, 52, 53]] type='5 * var * int64'>

## 小结

* `flatten(axis=n)` 压平指定的轴，`axis=None` 时变成一维并把 `None` 也去掉；`ravel()` 同样变成一维，但会保留 `None`。
* 从 NumPy 数组建出来的轴是固定长(regular)，从列表建出来的轴是可变长(var)，用 `from_regular()` 与 `to_regular()` 可以互相转换。
* 可变长的轴用 `pad_none()` 对齐长度、或者用 `clip=True` 截断之后，就能转成固定长。
* `unflatten()` 用给出的长度序列增加层次，指定 `axis` 时还能插入到已有的层次之中。
* 用 `None` 作下标会增加长度为 1 的轴，`singletons()` 则加上一条长度为 1 的可变长轴。
* 数组用 `concatenate()` 连接；记录可以增删字段，也能用 `unzip()` 与 `zip()` 拆开与合并。